In [1]:
pip install plotly

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.1.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# caminho + supressão de avisos
import os, warnings

# manipulação de dados
import pandas as pd
import numpy as np 

# conexão com o banco 
from sqlalchemy import create_engine

# aprendizado de maquina
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# visualização - dashboard
import plotly.graph_objects as go 
import plotly.subplots as make_subplots

In [3]:
warnings.filterwarnings('ignore')

In [4]:
# conectar o banco 
ENGINE_URL = ('mysql+pymysql://root:@localhost:3306/bolsa_familia')

engine = create_engine(ENGINE_URL, echo=False)

query = ''' 
        SELECT `MÊS COMPETÊNCIA`, `UF`, 
        COUNT(*) AS qtd_parcelas,
        AVG(`VALOR PARCELA`) AS valor_medio,
        SUM(`VALOR PARCELA`) AS valor_total  
        FROM bolsa_familia
        GROUP BY `MÊS COMPETÊNCIA`, `UF`
        '''
df = pd.read_sql(query, engine)

In [8]:
# EDA 
df_uf = df.groupby('UF').agg(media_valor = ('valor_medio', 'mean'),total_parcelas = ('qtd_parcelas', 'sum')).reset_index()


In [6]:
# calcular 
serie = df_uf['media_valor']
q1, q2, q3 = np.percentile(serie, [25, 50, 75])
iqr = q3 - q1
print(f'Media: {serie.mean():.2f} // Mediana: {serie.median():.2f}')
print(f'Q1: {q1}, Q3: {q3}, IQR: {iqr}')
print(f'Assimetria: {serie.skew():.3f}')
print(f'Curtose: {serie.kurt():.3f}')


Media: 675.99 // Mediana: 665.70
Q1: 659.6646461509814, Q3: 681.7392755192981, IQR: 22.074629368316664
Assimetria: 1.362
Curtose: 0.870


In [10]:
# aprendizado // agrupamento // não supervisionado 
features = df_uf[['media_valor','total_parcelas']].values

# boas práticas para aprendizado de maquina 
scaler = StandardScaler()
features_norm = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df_uf['cluster'] = kmeans.fit_predict(features_norm)
print(df_uf.groupby('cluster')[['UF', 'media_valor']].apply(lambda x: x).to_string())

            UF  media_valor
cluster                    
0       5   CE   659.898907
        10  MG   652.718213
        15  PE   662.686574
        18  RJ   658.468793
1       0   AC   716.495670
        2   AM   725.013274
        3   AP   715.441875
        21  RR   735.045058
2       1   AL   676.284181
        6   DF   665.701533
        7   ES   661.125371
        8   GO   663.684937
        11  MS   677.371649
        12  MT   680.291540
        14  PB   664.590413
        16  PI   665.985665
        17  PR   657.916298
        19  RN   655.405382
        20  RO   676.544558
        22  RS   671.001789
        23  SC   659.430385
        24  SE   661.850991
        26  TO   683.187011
3       9   MA   693.613220
        13  PA   695.759990
4       4   BA   658.331147
        25  SP   657.795862
